# 🎯 [OPS04_M01] 샤헤드 자폭드론 및 순항미사일 복합 비행 특성 분석 및 포화경보 데이터 규격 정의
### Google Colab 실습 환경 (작전·전투 수행 / Level 3)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/OPS04/ops04_m01_solution.ipynb)

---

## 📌 1. 프로젝트 및 미션 개요

| 항목 | 내용 |
| :--- | :--- |
| **과정 분류** | 작전·전투 수행 (Level 3) |
| **미션 ID** | OPS04_M01 |
| **미션 주제** | 샤헤드 자폭드론 및 순항미사일 복합 비행 특성 분석 및 포화경보 데이터 규격 정의 |
| **소요 시간** | 약 1.5시간 |
| **핵심 산출물** | `data/schema/ops04_input_schema.json, docs/ops04_data_spec.md` |
| **환경 사양** | Python 3.10+, numpy, pandas, matplotlib, scipy, scikit-learn |

## 📂 2. 표준 산출물 디렉토리 체계 및 실습 팁
- **방법 A (`%%writefile` 저장 추천):** 코드 셀 첫 줄에 `%%writefile <산출물 경로>`(예: `%%writefile src/models/ops04_baseline_model.py`)를 추가해 해당 폴더에 모듈을 저장할 수 있습니다.
- **방법 B (셀 단위 논리 대응 인정):** 별도 파일 추출 없이 노트북 셀 단위로 독립 실행·검증하고 미션 10 통합을 완료해도 정상 인정됩니다.

> **⚠️ [교육용 가상 데이터셋 및 HITL 원칙]**
> - 본 노트북의 모든 좌표·비행체 유형·수치는 보안 규정을 준수하여 **교육용 가상 데이터로 생성**됩니다.
> - 실제 무기 제원·사거리·부대 위치·군사 기밀 데이터는 일체 사용하지 않습니다.
> - 산출되는 대응 순위는 지휘관 승인을 위한 **'권고'**이며 자동 승인·자동 실행은 없습니다.

---

In [ ]:
# ============================================================================
# [Google Colab 환경 설정] 필수 라이브러리 설치 및 임포트
# ============================================================================
!pip install -q numpy pandas matplotlib scipy scikit-learn

import os
import sys
import time
import math
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# 그래프 영문 폰트 기본 설정 (환경 독립성 보장)
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

print(f"실행 환경 준비 완료 (Python {sys.version.split()[0]})")


---

## 🎯 미션 01. 샤헤드 자폭드론 및 순항미사일 복합 비행 특성 분석 및 포화경보 데이터 규격 정의
**소요시간**: 약 1.5시간 | **이번 산출물**: `data/schema/ops04_input_schema.json, docs/ops04_data_spec.md`

### 📋 미션 개요
**포화경보 상황에서 센서가 송신하는 항적 메시지의 표준 스키마(15개 필드)를 정의하고, 오류 데이터를 사전에 차단하는 입력 검증기를 구현합니다.**

- **배경 및 필요성:** 저속 자폭드론과 고속 순항미사일이 동시 침투하는 복합 포화 상황에서는 센서 메시지의 형식 오류나 누락이 후속 위협 판단 오류로 직결되므로, 엄격한 데이터 계약(Schema) 수립이 필수적입니다.
- **핵심 과업:** 항적 ID·시각·위치·고도·속도·방위·접근 방향·유형 추정·추상 위협 가중 등급(1~3)과 전술 파라미터(기상·위치오차·통신지연)를 정의하고, 정상 샘플 20건 검증 및 5종 오류(형식·허용값·범위·누락·미정의 필드 `warhead_kg`) 주입 검출 테스트를 수행합니다.
- **학습 목표 (습득 역량):** 복합 공중위협 항적의 핵심 전술 속성 모델링 역량, 실제 무기 제원을 추상화한 보안 안전 스키마 설계 능력, 결함 주입 기반 데이터 유효성 검증 역량을 체득합니다.
- **검증 및 산출물:** 스키마 필드 정의 완성도 100%, 정상 샘플 오류 0건 및 주입 오류 5/5 검출을 통과한 입력 스키마(`data/schema/ops04_input_schema.json`)와 규격 명세서(`docs/ops04_data_spec.md`)를 산출합니다.

### 🎓 학습 목표 (습득 역량)
**[습득 역량]** 복합 공중위협(저속 무인기형·고속 순항형·기만체)의 비행 특성을 보안 안전한 추상 데이터 구조로 모델링하고, JSON Schema 기반 입력 규격 설계 및 결함 주입 무결성 검증 역량을 체득합니다.

**[핵심 과업]** 항적 ID·센서·시각·위치·고도·속도·방위·접근 방향·유형 추정·추상 위협 가중 등급(1~3)과 전술 파라미터(기상·위치오차·통신지연) 필드를 정의하고, 가상 샘플 20건 검증 및 5종 오류 주입 테스트 후 데이터 명세서를 작성합니다.

- **전술 데이터 모델링 및 보안 추상화 역량:** 실제 탄두 위력 등 민감 무기 제원을 사용하지 않고 추상 '위협 가중 등급(1~3)'과 전술 환경 변수로 치환해 표현하는 보안 안전 데이터 설계 원리를 익힙니다.
- **표준 입력 스키마(JSON Schema) 설계 능력:** 다종 센서 경보 메시지의 15개 필수·선택 필드, 데이터 타입, 허용값(enum) 및 수치 범위를 엄격하게 제약하는 검증 규격 작성 기술을 습득합니다.
- **결함 주입(Fault Injection) 기반 무결성 검증 역량:** 형식·허용값·범위·누락·미정의 필드(`warhead_kg` 등) 오류 5종을 의도적으로 주입해 검증기의 차단 성능을 입증하고 명세서로 문서화하는 능력을 기릅니다.

### 📖 미션 배경 및 시나리오
#### 미션 배경 및 당위성
샤헤드-136 자폭드론 및 순항미사일 동시 침투 전술을 분석하여 비행 속도, 고도, 탄두 위력, 표적 접근 방향 등 포화경보 핵심 속성 데이터 규격을 정의한다. 실제 전술 현장에서는 단순 탐지에 그치지 않고 신속한 위협 판단과 신뢰도 높은 의사결정이 필수적이므로, **포화경보 상황에서 센서가 보내는 항적 메시지의 핵심 속성을 정의하고, 잘못된 메시지를 걸러내는 검증기를 만든다.** 과업을 완수하여 오판을 방지하고 작전 대응력을 극대화해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **항적 ID, 센서, 시각, 위치, 고도, 속도, 방위, 접근 방향, 유형 추정, 위협 가중 등급 필드를 정하고 기상·위치오차·통신지연을 전술 파라미터로 보완 수행하고 검증하는** 실습을 체계적으로 수행합니다.

학습자는 실습가이드에 제시된 총 4개 세부 활동(기본 기능 구현 ➔ 결과 지표 검증 ➔ 파라미터 튜닝 및 도전과제)을 순차적으로 진행하며 핵심 판정 기준을 달성합니다.

**이번 결과물:** `data/schema/ops04_input_schema.json 및 docs/ops04_data_spec.md` 형태의 결과물을 완성한다.

### 🤖 단계별 수행 및 AI 협업 가이드
#### Step 1. 미션 1 코드를 실행해 정상 샘플 20건의 검증 결과를 확인한다.
#### 이 단계의 역할
미션 1의 스키마 정의(`SCHEMA`, `REQUIRED`)와 `validate()` 함수를 실행해 정상 가상 샘플 20건이 오류 0건으로 통과하는지 확인합니다.

#### 해야 할 일

- 항적 ID, 센서, 시각, 위치, 고도, 속도, 방위, 접근 방향, 유형 추정, 추상 위협 가중 등급(1~3) 및 전술 파라미터(기상·위치오차·통신지연) 15개 필드의 타입과 허용 범위를 파악한다.
- 생성형 AI를 Co-pilot으로 활용해 가상 항적 메시지를 생성하는 `to_message()`와 15개 필드 규격을 검사하는 `validate()` 함수 구조를 검토·보완한다.
- 정상 샘플 20건을 `validate()`에 통과시켜 스키마 오류 0건과 미정의 필드 차단 여부를 확인한다.

#### AI 협업 가이드 (프롬프트 & 검증 지침)

  - **프롬프트 질의 예시:** "포화경보 항적 메시지 15개 필드(track_id, sensor, ts, x_km, y_km, alt_m, speed_mps, heading_deg, approach_dir, type_est, threat_grade 1~3, weather, pos_err_m, comm_delay_s 등)를 검증하는 Python validate(msg) 함수를 작성해줘. 실제 탄두 중량 같은 미정의 필드는 거부해야 해."

  - **AI 결과 검증 포인트:** AI가 생성한 코드에 실제 무기 제원(탄두 위력·사거리 등)이나 부대명이 포함되지 않았는지 확인하고, `REQUIRED` 15개 필드의 누락·타입 불일치·허용값 이탈이 정확히 차단되는지 교차 검증합니다.

#### 핵심 확인 지표

> 정상 샘플 오류 0건, 주입 오류 5/5 검출, 스키마 15개 필드 정의 완성도 100%, 미정의 필드(warhead_kg) 차단

#### 학습자가 제출해야 하는 내용

- 미션 1 코드 실행 결과 및 정상 샘플 20건 검증 로그
- AI 코파일럿 질의 프롬프트 및 스키마 제약조건 검증 기록

#### Step 2. 오류 주입 5종이 각각 어떤 규칙에 걸렸는지 표로 정리한다.
#### 이 단계의 역할
형식·허용값·범위·누락·미정의 필드(`warhead_kg`) 등 의도적 오류 5종을 주입해 검증기가 각각 올바른 규칙으로 차단하는지 검증합니다.

#### 해야 할 일

- 현장 센서 통신에서 발생 가능한 5종 결함(타입 형식 오류, 허용 enum 이탈, 물리 범위 초과, 필수 필드 누락, 미정의 보안 민감 필드 추가) 테스트 케이스를 준비한다.
- AI에게 5종 오류 주입 테스트 코드를 요청해 각 오류가 어떤 검증 분기에서 포착되는지 단위 테스트를 구성한다.
- 실행 결과에서 5종 오류가 모두 검출되었는지 확인하고, 오류 유형별 차단 사유를 표로 정리한다.

#### AI 협업 가이드 (프롬프트 & 검증 지침)

  - **프롬프트 질의 예시:** "앞서 만든 validate(msg) 함수에 대해 1) 문자열/숫자 형식 오류, 2) 허용값 이탈, 3) 고도/속도 범위 이탈, 4) 필수 키 누락, 5) 미정의 필드(warhead_kg) 추가 등 5가지 오류를 주입해 모두 예외/오류 사유를 표로 출력하는 테스트 코드를 작성해줘."

  - **AI 결과 검증 포인트:** AI가 작성한 테스트가 단순히 예외 발생 여부만 세지 않고, 주입한 5종 오류 각각이 의도한 검증 규칙(예: `warhead_kg` 미정의 필드 차단)에 정확히 매칭되는지 확인합니다.

#### 핵심 확인 지표

> 주입 오류 5/5 검출(100%), 미정의 필드(warhead_kg) 차단 확인

#### 학습자가 제출해야 하는 내용

- 오류 주입 5종별 검증 규칙 매핑표 및 단위 테스트 로그
- AI 생성 테스트 케이스 검증 및 수정 메모

#### Step 3. 데이터 명세 요약표에서 기상·위치오차·통신지연 필드가 포함되었는지 확인한다.
#### 이 단계의 역할
기상·위치오차·통신지연 전술 파라미터와 보안 추상화 원칙이 데이터 명세서(`docs/ops04_data_spec.md`)에 빠짐없이 반영되었는지 검증합니다.

#### 해야 할 일

- 생성된 `SCHEMA`와 데이터 명세 요약표를 대조해 기상·위치오차·통신지연 전술 파라미터의 단위와 허용 범위를 확인한다.
- AI에게 전술 파라미터 포함 여부와 추상 '위협 가중 등급(1~3)' 대체 원칙이 드러나는 마크다운 명세서 템플릿 정리를 요청한다.
- 명세서 내 모든 필드 수(15개)와 JSON Schema(`data/schema/ops04_input_schema.json`)의 일치 여부를 최종 대조한다.

#### AI 협업 가이드 (프롬프트 & 검증 지침)

  - **프롬프트 질의 예시:** "포화경보 스키마 15개 필드와 전술 파라미터(기상, 위치오차, 통신지연), 그리고 실제 무기 제원을 추상 위협 가중 등급(1~3)으로 대체한 보안 원칙을 포함한 ops04_data_spec.md 명세서 요약표를 마크다운으로 작성해줘."

  - **AI 결과 검증 포인트:** AI가 명세서에 존재하지 않는 임의의 필드나 실제 무기 제원(사거리·탄두중량 등)을 환각으로 추가하지 않았는지 코드의 `SCHEMA` 정의와 1:1 대조합니다.

#### 핵심 확인 지표

> 스키마 15개 필드 정의 완성도 100%, 기상·위치오차·통신지연 전술 파라미터 명세 완료

#### 학습자가 제출해야 하는 내용

- 전술 파라미터 포함 데이터 명세 요약표(`docs/ops04_data_spec.md` 초안)
- AI 작성 명세서와 실제 SCHEMA 대조 검증 메모

#### Step 4. 도전 과제 및 파라미터 튜닝 최적화
#### 이 단계의 역할
기본 스키마(`SCHEMA`)에 표적 군집 수(`group_size`, 1~50) 필드를 확장하고 경계값 이탈 차단을 검증하는 심화 튜닝을 수행합니다.

#### 해야 할 일

- `SCHEMA`에 `group_size`(정수, 허용 범위 1~50) 필드를 추가하고 기존 정상 샘플 20건에 기본값을 반영한다.
- AI에게 `group_size` 경계값(0, 1, 50, 51, 실수형) 테스트 케이스 생성을 요청해 경계 조건 동작을 확인한다.
- 기존 5종 오류 검출 및 정상 샘플 통과 성능이 유지되는지 회귀 검증하고 변경 전후 분석 메모를 작성한다.

#### AI 협업 가이드 (프롬프트 & 검증 지침)

  - **프롬프트 질의 예시:** "기존 SCHEMA와 validate()에 group_size(int, 1~50) 필드를 추가하고, 경계값(0, 1, 50, 51) 입력 시 범위 이탈이 정확히 검출되는지 확인하는 비교 테스트 코드를 만들어줘."

  - **AI 결과 검증 포인트:** 신규 필드 추가 시 기존 15개 필드의 검증 규칙과 정상 샘플 통과율(오류 0건)이 깨지지 않는지(Regression 방지) 반드시 확인합니다.

#### 핵심 확인 지표

> group_size 범위 이탈(0, 51 등) 100% 검출, 기존 정상 샘플 20건 오류 0건 유지

#### 학습자가 제출해야 하는 내용

- 파라미터 확장 전후 비교 분석 메모 및 경계값 테스트 결과
- AI 프롬프트 질의 및 회귀 테스트 검증 기록

### 🔍 평가 및 검증 기준 (PASS 조건)
#### 미션 판정
**평가 구분:** 필수

**핵심 평가영역:** 작전·전투 수행 실무 및 러시아-우크라이나 전쟁 전장 환경과의 군사적 부합성

#### 미션 PASS 조건
- 정량 기준: 스키마 필드 정의 완성도 100%, 샘플 데이터 검증 오류 0건
- 진행 기준: 입력 데이터 유효성 검증 스크립트 통과 및 명세서 문서화 완료
- 검증 방법: JSON Schema 검증 도구 및 샘플 데이터 파싱 단위 테스트
- AI 검증 기준: 포화경보 15개 필드 스키마 정의와 5종 오류(warhead_kg 포함) 차단 로직을 구조화된 AI 프롬프트로 구현하고 보안·제약조건을 HITL로 교차 검증 완료
#### 채점자 판정 메모

> 단순 코드 실행 여부뿐만 아니라 AI 프롬프트 설계의 구체성, AI 산출물의 오류·보안 위반을 인간 검토(HITL)로 식별·수정한 기록, 그리고 출력 수치의 군 실무 부합성을 종합 채점한다.

In [ ]:
"""포화경보 입력 메시지의 데이터 규격(JSON 스키마)을 정의하고, 가상 샘플 20건 검증과 오류 5종 주입 검출로 검증기를 확인한 뒤 데이터 명세서를 출력한다."""
import json
import numpy as np
import pandas as pd

# ---------------- [교육용가상자료] 가상 방어지역·가상 비행체 생성기 ----------------
# 좌표는 가상 지휘소를 원점으로 한 km 단위 평면이다. 비행체 유형은 추상 범주이며
# 속도·고도·반사강도는 학습용 임의 범위로, 실제 무기 제원·사거리·탄도와 무관하다.
ASSETS = pd.DataFrame([["A1", "CP(virtual)", 0.0, 0.0, 1.0],
                       ["A2", "Radar(virtual)", 4.0, -3.0, 0.8],
                       ["A3", "Depot(virtual)", -5.0, 4.0, 0.7],
                       ["A4", "Power(virtual)", 7.0, 6.0, 0.5]], columns=["asset", "name", "x", "y", "value"])
TYPES = {  # 유형: (위협 여부, 속도 m/s 범위, 고도 m 범위, 출현 비율)
    "uav_type": (True, (30, 60), (100, 800), 0.42),      # 저속 무인기형(가상)
    "fast_type": (True, (150, 250), (50, 200), 0.16),    # 고속 순항형(가상)
    "decoy": (False, (30, 60), (100, 800), 0.16),        # 기만체(가상)
    "civil": (False, (120, 220), (3000, 9000), 0.10),    # 민항기(가상)
    "bird": (False, (8, 18), (50, 400), 0.16),           # 조류 군집
}
TTI_URGENT = 600      # 긴급 판정 기준: 가상 자산 도달 예상시간 600초 이내
CPA_KM = 1.5          # 자산 상공 통과 판정 반경(가상)


def make_raid(seed, n=80, spoof=0):
    """가상 포화 상황 1회분(비행체 n개 + 스푸핑 가짜 항적 spoof개)을 만든다."""
    rng = np.random.default_rng(seed)
    names = list(TYPES)
    rows = []
    for i in range(n + spoof):
        typ = "spoof" if i >= n else str(rng.choice(names, p=[TYPES[k][3] for k in names]))
        threat, sp, al, _ = TYPES["decoy" if typ == "spoof" else typ]
        brg = np.radians(rng.uniform(0, 360)); dist = rng.uniform(6, 60)
        x, y = dist * np.sin(brg), dist * np.cos(brg)
        spd = rng.uniform(*sp); alt = rng.uniform(*al)
        aim = int(rng.integers(0, len(ASSETS)))
        if typ in ("uav_type", "fast_type", "decoy", "spoof"):      # 가상 자산 방향(작은 방위 오차)
            ax_, ay_ = ASSETS.loc[aim, ["x", "y"]]
            hd = np.arctan2(ax_ - x, ay_ - y) + np.radians(rng.normal(0, 0.6))
        else:
            hd = rng.uniform(0, 2 * np.pi); aim = -1
        vx, vy = spd * np.sin(hd) / 1000, spd * np.cos(hd) / 1000          # km/s
        sig = {"uav_type": ((0.2, 0.5), (0.02, 0.12), [0, .1, .5, .4]), "fast_type": ((0.1, 0.4), (0.02, 0.12), [0, .2, .4, .4]),
               "decoy": ((0.3, 0.8), (0.08, 0.35), [0, .55, .4, .05]), "spoof": ((0.3, 0.8), (0.05, 0.3), [0, .7, .3, 0]),
               "civil": ((0.7, 1.0), (0.01, 0.05), [0, 0, .1, .9]), "bird": ((0.01, 0.12), (0.1, 0.4), [0, .6, .4, 0])}[typ]
        jump = spd * rng.uniform(0.95, 1.2) if typ != "spoof" else spd * rng.uniform(4, 15)
        true_tti = np.inf
        if threat:
            d_aim = np.hypot(ASSETS.loc[aim, "x"] - x, ASSETS.loc[aim, "y"] - y)
            true_tti = d_aim * 1000 / spd
        rows.append({"id": f"T{seed % 100:02d}{i:03d}", "type": typ, "x": x, "y": y, "vx": vx, "vy": vy, "alt": alt, "spd": spd,
                     "rcs": rng.uniform(*sig[0]), "rcs_var": rng.uniform(*sig[1]), "n_sensors": int(rng.choice(4, p=sig[2])),
                     "max_jump": jump, "civ_iff": int(typ == "civil" and rng.random() < 0.95),
                     "aim": aim, "threat": threat, "true_tti": true_tti,
                     "urgent": bool(threat and true_tti <= TTI_URGENT)})
    df = pd.DataFrame(rows)
    df["true_prio"] = [ASSETS.loc[a, "value"] * np.exp(-t / 300) if th else 0.0
                       for a, t, th in zip(df["aim"], df["true_tti"], df["threat"])]
    return df


def geometry(df):
    """각 항적과 가상 자산 사이 최근접점(CPA) 거리와 도달 예상시간(TTI, s)을 계산한다."""
    px, py = df["x"].to_numpy()[:, None], df["y"].to_numpy()[:, None]
    vx, vy = df["vx"].to_numpy()[:, None], df["vy"].to_numpy()[:, None]
    rx, ry = ASSETS["x"].to_numpy()[None] - px, ASSETS["y"].to_numpy()[None] - py
    v2 = vx ** 2 + vy ** 2 + 1e-12
    t = (rx * vx + ry * vy) / v2                                  # CPA 도달 시각(s)
    cpa = np.hypot(rx - vx * t, ry - vy * t)
    tti = np.where((t > 0) & (cpa < CPA_KM), t, np.inf)
    return cpa, tti

# 포화경보 입력 메시지 스키마. 실제 무기 제원(탄두 위력·사거리 등) 대신 추상 '위협 가중 등급'을 쓴다.
SCHEMA = {
    "track_id":      {"type": str, "rule": "T+숫자 5자리", "check": lambda v: v[0] == "T" and v[1:].isdigit() and len(v) == 6},
    "sensor_id":     {"type": str, "allowed": ["RDR-1", "RDR-2", "EO-1", "ACU-1"]},
    "timestamp_ms":  {"type": int, "range": (0, 10 ** 13)},
    "x_km":          {"type": float, "range": (-100, 100)},
    "y_km":          {"type": float, "range": (-100, 100)},
    "alt_m":         {"type": float, "range": (0, 15000)},
    "speed_mps":     {"type": float, "range": (0, 400)},
    "heading_deg":   {"type": float, "range": (0, 360)},
    "approach_dir":  {"type": str, "allowed": ["N", "NE", "E", "SE", "S", "SW", "W", "NW"]},
    "class_hint":    {"type": str, "allowed": ["uav_type", "fast_type", "unknown", "civil", "bird"]},
    "threat_grade":  {"type": int, "range": (1, 3)},          # 위협 가중 등급(가상, 1 낮음 ~ 3 높음)
    "rcs":           {"type": float, "range": (0, 1)},        # 정규화 반사강도(가상)
    "weather":       {"type": str, "allowed": ["clear", "rain", "fog", "snow"]},
    "pos_error_m":   {"type": float, "range": (0, 2000)},     # 보완 전술 파라미터: 위치 오차
    "comm_delay_ms": {"type": int, "range": (0, 5000)},       # 보완 전술 파라미터: 통신 지연
}
REQUIRED = list(SCHEMA)


def validate(msg):
    errs = []
    for f in REQUIRED:
        if f not in msg or msg[f] is None:
            errs.append(f"{f}: 누락"); continue
        spec, v = SCHEMA[f], msg[f]
        if spec["type"] is float and isinstance(v, int) and not isinstance(v, bool):
            v = float(v)
        if not isinstance(v, spec["type"]):
            errs.append(f"{f}: 형식 오류({type(v).__name__})"); continue
        if "allowed" in spec and v not in spec["allowed"]:
            errs.append(f"{f}: 허용값 아님({v})")
        if "range" in spec and not (spec["range"][0] <= v <= spec["range"][1]):
            errs.append(f"{f}: 범위 초과({v})")
        if "check" in spec and not spec["check"](v):
            errs.append(f"{f}: 규칙 위반({v})")
    extra = set(msg) - set(SCHEMA)
    if extra:
        errs.append(f"정의되지 않은 필드: {sorted(extra)}")
    return errs


def to_message(r, rng):
    hd = (np.degrees(np.arctan2(r.vx, r.vy)) + 360) % 360
    frm = (hd + 180) % 360
    hint = {"uav_type": "uav_type", "fast_type": "fast_type", "civil": "civil", "bird": "bird"}.get(r.type, "unknown")
    return {"track_id": r.id, "sensor_id": str(rng.choice(SCHEMA["sensor_id"]["allowed"])), "timestamp_ms": int(1.7e12 + rng.integers(0, 60000)),
            "x_km": round(float(r.x), 2), "y_km": round(float(r.y), 2), "alt_m": round(float(r.alt), 0), "speed_mps": round(float(r.spd), 1),
            "heading_deg": round(float(hd), 1), "approach_dir": ["N", "NE", "E", "SE", "S", "SW", "W", "NW"][int(((frm + 22.5) % 360) // 45)],
            "class_hint": hint, "threat_grade": 3 if r.type == "fast_type" else (2 if r.threat else 1), "rcs": round(float(r.rcs), 2),
            "weather": str(rng.choice(["clear", "rain", "fog", "snow"])), "pos_error_m": round(float(rng.uniform(20, 300)), 0),
            "comm_delay_ms": int(rng.integers(50, 800))}


rng = np.random.default_rng(1)
raid = make_raid(11, 20)
samples = [to_message(r, rng) for r in raid.itertuples()]
normal_err = sum(len(validate(m)) for m in samples)
print(f"[정상 샘플 검증] {len(samples)}건, 오류 {normal_err}건")
print("샘플 1건:", json.dumps(samples[0], ensure_ascii=False)[:150], "...")

bad = [dict(samples[0]) for _ in range(5)]
bad[0]["speed_mps"] = "fast"                 # 형식 오류
bad[1]["sensor_id"] = "RDR-9"                # 허용값 오류
bad[2]["alt_m"] = -50.0                      # 범위 오류
del bad[3]["comm_delay_ms"]                  # 누락(통신 지연)
bad[4]["warhead_kg"] = 50                    # 정의되지 않은 필드(실제 제원 입력 차단)
kinds = ["형식", "허용값", "범위", "누락", "미정의 필드"]
print("\n[오류 주입 검출]")
detected = 0
for k, m in zip(kinds, bad):
    e = validate(m); detected += bool(e)
    print(f"  {k:7s} -> {'검출' if e else '미검출'}: {e[0] if e else ''}")

print("\n[데이터 명세 요약]")
spec = pd.DataFrame([{"필드": f, "형식": s["type"].__name__, "규칙": s.get("allowed", s.get("range", s.get("rule")))} for f, s in SCHEMA.items()])
print(spec.to_string(index=False))
complete = sum(all(k in s for k in ("type",)) and any(k in s for k in ("allowed", "range", "check")) for s in SCHEMA.values()) / len(SCHEMA)
print(f"\n스키마 필드 정의 완성도 {complete:.0%} (목표 100%) -> {'달성' if complete == 1 else '미달'}")
print(f"샘플 검증 오류 {normal_err}건 (목표 0건) -> {'달성' if normal_err == 0 else '미달'}, 주입 오류 {detected}/5 검출")
print("참고: 실제 무기 제원·사거리 필드는 두지 않으며, 미정의 필드는 검증 단계에서 차단한다.")


<details>
<summary><b>💡 [참고] 미션 01 예상 실행 결과 (Output)</b></summary>

```text
[정상 샘플 검증] 20건, 오류 0건
샘플 1건: {"track_id": "T11000", "sensor_id": "RDR-2", "timestamp_ms": 1700000030709, "x_km": 0.17, "y_km": -38.48, "alt_m": 204.0, "speed_mps": 30.9, "heading_ ...

[오류 주입 검출]
  형식      -> 검출: speed_mps: 형식 오류(str)
  허용값     -> 검출: sensor_id: 허용값 아님(RDR-9)
  범위      -> 검출: alt_m: 범위 초과(-50.0)
  누락      -> 검출: comm_delay_ms: 누락
  미정의 필드  -> 검출: 정의되지 않은 필드: ['warhead_kg']

[데이터 명세 요약]
           필드    형식                                          규칙
     track_id   str                                    T+숫자 5자리
    sensor_id   str                 [RDR-1, RDR-2, EO-1, ACU-1]
 timestamp_ms   int                         (0, 10000000000000)
         x_km float                                 (-100, 100)
         y_km float                                 (-100, 100)
        alt_m float                                  (0, 15000)
    speed_mps float                                    (0, 400)
  heading_deg float                                    (0, 360)
 approach_dir   str                [N, NE, E, SE, S, SW, W, NW]
   class_hint   str [uav_type, fast_type, unknown, civil, bird]
 threat_grade   int                                      (1, 3)
          rcs float                                      (0, 1)
      weather   str                    [clear, rain, fog, snow]
  pos_error_m float                                   (0, 2000)
comm_delay_ms   int                                   (0, 5000)

스키마 필드 정의 완성도 100% (목표 100%) -> 달성
샘플 검증 오류 0건 (목표 0건) -> 달성, 주입 오류 5/5 검출
참고: 실제 무기 제원·사거리 필드는 두지 않으며, 미정의 필드는 검증 단계에서 차단한다.
```
</details>
